# ReAct

⭐️ ReAct (Reasoning + Acting) ⭐️는 프린스턴 대학교 학자들이 구글 연구원들과 협력하여 개발한 프롬프트 엔지니어링 전략입니다. 이 전략은 ***LLM이 인간처럼 문제를 풀기 위해 먼저 생각하고, 그에 따라 행동***하는 방식을 모방합니다.

ReAct가 없는 모델의 경우 다음과 같은 문제가 발생할 수도 있습니다.
```
1. 사용자가 "안중근 의사랑 세종대왕이랑 이순신 장군 알려줘"라고 요청한 경우,
2. 일반적인 모델의 경우 안중근 의사, 세종대왕, 이순신 장군에 대한 정보를 하나씩 생성하여 보여줍니다.
3. 잘 생성하다가 안중근 의사, 세종대왕에 대한 정보만 생성하고, 이순신 장군에 대한 정보는 생성하지 않을 가능성이 존재합니다.
```

ReAct는 모델이 여러 외부 도구(검색 도구)를 사용할 수 있음을 전제로 하며, 도구의 결과를 바탕으로 다시 생각하고 행동을 이어갈 수 있게 합니다. 예를 들어, 다음과 같은 과정이 이뤄집니다:

	1.	사용자가 “안중근 의사랑 세종대왕이랑 이순신 장군 알려줘”라고 요청할 경우,
	2.	ReAct 모델은 먼저 문제를 생각하여 “안중근 의사부터 검색하고, 이후에 순차적으로 검색하는 것이 좋겠다”고 판단합니다.
	3.	그 후 행동으로 검색 도구를 호출하여 “안중근 의사”를 검색하고, 그 결과를 바탕으로 다시 생각과 행동을 반복해 나머지 인물들도 검색합니다.

이처럼 ReAct는 한번의 질문에 많은 컨텍스트가 혼재되어있는 경우 모델이 생각과 행동을 번갈아 수행하도록 하여, 복잡한 문제도 여러 단계를 거쳐 해결할 수 있도록 지원합니다. 이를 통해 LLM은 인간처럼 상황을 이해하고 문제를 해결하는 에이전트로 기능할 수 있게 됩니다.

> 프론트엔드의 ReAct와 다름

> CoT 방법 중 하나

In [ ]:
%%capture
# update or install the necessary libraries
# !pip install --upgrade openai
!pip install --upgrade openai
!pip install --upgrade langchain
!pip install --upgrade python-dotenv
!pip install google-search-results
!pip install langchain_community
!pip install ChatOpenAI

In [ ]:
import openai
import os

from langchain.chat_models import ChatOpenAI
from langchain.agents import load_tools
from langchain.agents import initialize_agent
from dotenv import load_dotenv
load_dotenv()

False

In [1]:
# 올바른 환경 변수 이름으로 설정
os.environ["OPENAI_API_KEY"] = "~~~"

os.environ["SERPAPI_API_KEY"] = "~~~"

NameError: name 'os' is not defined

# ReAct 준비하기

In [ ]:
# ReAct를 적용할 모델
llm = ChatOpenAI(
    model_name='gpt-4o',
    temperature=0
)

<ipython-input-6-105e96af27b5>:2: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [ ]:
# 도구 설정 : 구글링 API인 serpapi와 산수 계산을 위한 llm-math 도구를 가져옴

tools = load_tools(["serpapi", "llm-math"], llm=llm)

tools에 대한 커스터마이징도 가능합니다. https://python.langchain.com/v0.1/docs/modules/tools/custom_tools/ 를 참고하세요

In [2]:
tools

NameError: name 'tools' is not defined

In [17]:
# model과 모델이 사용할 도구를 합하여 agent를 생성
agent = initialize_agent(
    tools,
    llm,
    agent='zero-shot-react-description',
    verbose=True,
    handle_parsing_errors=True
)

# ReAct 기반의 답변 생성

In [18]:
response = agent.run("오늘 날짜와 어제 나스닥 마감 지수가 몇이였는지 알려줘.")



> Entering new AgentExecutor chain...
To answer this question, I need to find out today's date and the closing value of the Nasdaq index for yesterday. 

Action: Search
Action Input: "today's date"

Observation: Tuesday, October 15, 2024
Thought:I have today's date. Now, I need to find the closing value of the Nasdaq index for yesterday, which would be October 14, 2024.

Action: Search
Action Input: "Nasdaq closing value October 14, 2024"

Observation: {'type': 'organic_result', 'title': 'NASDAQ Composite (^IXIC)', 'source': 'Yahoo Finance'}
Thought:It seems I couldn't retrieve the specific closing value from the search results. I will try another search to find the Nasdaq closing value for October 14, 2024.

Action: Search
Action Input: "Nasdaq closing value October 14, 2024 Yahoo Finance"

Observation: ['Oct 14, 2024, 18,426.66, 18,547.92, 18,423.60, 18,502.69, 18,502.69, 4,894,710,000 ... Copyright © 2024 Yahoo. All rights reserved. POPULAR QUOTES. Dow Jones.', 'The tech-heavy Nas

In [19]:
response

"Today's date is October 15, 2024, and the Nasdaq closing value for October 14, 2024, was 18,342.94."

In [20]:
response = agent.run("레오나르도 다빈치의 작품의 개수와 레오나르도 디카프리오의 여자친구의 나이를 더해줘")



> Entering new AgentExecutor chain...
레오나르도 다빈치의 작품 개수와 레오나르도 디카프리오의 여자친구의 나이를 알아야 합니다.

1. 레오나르도 다빈치의 작품 개수를 알아보겠습니다.
2. 레오나르도 디카프리오의 여자친구의 나이를 알아보겠습니다.

Action: Search
Action Input: "레오나르도 다빈치 작품 개수"

Observation: ['1501년 <성 모자와 성 안나> 제단화를 그린 것을 시작으로 수많은 연구 스케치와 관찰을 남겼다. 헬리콥터나 인체비례도 등 잘 알려진 다빈치의 과학자/ ...', '거장 레오나르도 다빈치 11작품 소개 · ①수태고지 · ② 그리스도의 세례 · ③동방박사의 예배 · ④ 암굴의 성모 · ⑤흰 족제비를 안은 여인 · ⑥최후의 만찬 <복구 전>(상), < ...', '그는 평생 기술과 과학 그리고 예술을 하나로 융합하려는 창의적인 노력을 끓임없이 시도하였다. 그의 여러 작품 중 오늘날 가장 유명한 것은 모나리자이다. 모나리자 ...', '레오나르도 다빈치 화가는 완성된 작품보다 미작품이 많습니다. 대표적으로는 완성작은 모자리자, 최후의 만찬, 성애, 세명의 성모 등등 입니다. 작품하나 ...', '레오나르도 다빈치 38 작품모음. 레오나르도 다빈치 작품모음보면 레오나르도 다 빈치의 천재성과. 작품에 대한 열정이 느껴집니다.', '1896년까지는 스승인 베로키오의 작품으로 생각되었으나, 현재는 다빈치의 작품으로 받아들여진다. \u200b. 다빈치는 20세가 되던 1472년에 피렌체의 예술가 ...', '레오나르도 다빈치 하면 생각나는 작품은.... 모나리자, 최후의 만찬, 그리고 홀딱 벗고 대자로 누워있는 아저씨 비트루비안 맨... 그리고.', '위대한 과학자이자 예술가 레오나르도 다빈치의 작품이 21세기에 갑자기 발견된다면 어떤 일이 벌어질까요? 그림 발견과 동시에 수백억원 넘는 가격을 ...', '수레의 바퀴가 한번 돌아갈 때마다 민첩한 장치가 돌멩이 하나를 바구니에 떨어트린다

In [22]:
response

'45'

# ReAct Agent + RAG 결합
이번에는 RAG를 토대로 ReACT하는 방법에 대해 알아봅니다. 검색 도구를 vectorDB로 지정하여 vectorDB 내에서 정보를 검색하도록 유도하면 됩니다.

In [26]:
!pip install --quiet langchain tiktoken chromadb pymupdf sentence-transformers openai pymupdf

In [27]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain.tools.retriever import create_retriever_tool
from langchain import hub
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain.agents import initialize_agent

In [29]:
# PDF 파일로드
loader = PyMuPDFLoader('/content/sample_data/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf')
data = loader.load()

# 청킹
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=512, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(data)

In [30]:
# 임베딩 설정
embedding_model = "BAAI/bge-m3"

embd = HuggingFaceEmbeddings(
    model_name=embedding_model
)

<ipython-input-30-9690633ae256>:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embd = HuggingFaceEmbeddings(
/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/t

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [34]:
# 벡터 스토어로 적재
vectorstore = Chroma.from_documents(persist_directory='/content/vector',
    documents=doc_splits,
    embedding=embd,
)

vectorstore_retriever = vectorstore.as_retriever()

In [32]:
vectorstore_retriever.get_relevant_documents("세계 인공지능 시장 규모 설명")

<ipython-input-32-195d5e321be2>:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  vectorstore_retriever.get_relevant_documents("세계 인공지능 시장 규모 설명")


[Document(metadata={'author': 'kmd kdy', 'creationDate': "D:20220729090316+09'00'", 'creator': 'Hancom PDF 1.3.0.538', 'file_path': '/content/sample_data/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf', 'format': 'PDF 1.4', 'keywords': '', 'modDate': "D:20220729090316+09'00'", 'page': 9, 'producer': 'Hancom PDF 1.3.0.538', 'source': '/content/sample_data/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf', 'subject': '', 'title': '', 'total_pages': 18, 'trapped': ''}, page_content='세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 \n정도로 압도적으로 높음\n[세계 인공지능 시장규모]\n                                                            (단위: 억 달러, 괄호는 YoY %)\n구분\n2020\n2021\n2022\n2023\n2024\n2025\nCAGR\n(2020-2025)\n인공지능\n398.4\n553.3\n769.7\n1,134.3\n1,498.9\n2,223.7\n41.0%\n\u3000\n(38.9)\n(39.1)\n(47.4)\n(32.1)\n(48.4)\n  \n소프트웨어\n311.8\n432.3\n600.3\n882.4\n1,164.6\n1,723.5\n40.8%\n\u3000\n(38.6)\n(38.8)\n(47.0)\n(32.0)\n(48.0)\n  \n서비스\n55.7\n78.0\n109.3\n162.6\n216.0\n32

In [35]:
# 툴 생성
vectorstore_search_tool = create_retriever_tool(
    retriever=vectorstore_retriever,
    name="ICT_industry_search", # 도구를 선택 하는 단계에서 사용할 이름
    description="This PDF provides information about trends in the ICT industry, including AI."
)

In [36]:
tools = [ vectorstore_search_tool ]

## React Prompt 살펴보기
툴을 생성하면 일반적으로 ReACT 프롬프트에 다음과 같이 기록됩니다.
```
다음 질문에 최선을 다해 답변하세요. 당신이 사용할 수 있는 도구는 다음과 같습니다.:

ICT_industry_search: This PDF provides information about trends in the ICT industry, including AI.

질문: 당신이 답해야 할 입력 질문
생각: 무엇을 해야 할지 생각하세요
행동: 취해야 할 행동을 선택하세요. [ ICT_industry_search ] 중 하나여야 합니다.
행동 입력: 행동에 대한 입력을 작성하세요.
관찰: 행동의 결과를 작성하세요
...(이 생각/행동/행동 입력/관찰 단계는 여러 번 반복될 수 있습니다)
생각: 이제 최종 답변을 알겠습니다.
최종 답변: 원래 질문에 대한 최종 답변을 한국어로 작성하세요.

시작하세요!
```

즉 ReACT에서 기본적으로 사용할 Tool에 대한 설명(name과 Description)을 한 후 이 툴을 이용해 **행동**하도록 유도합니다.

여러 개의 툴을 사용하는 경우 다음과 같은 프롬프트가 기록됩니다. 계산기용 `Calculator` tool이 추가 됐다고 가정합니다.

툴을 생성하면 일반적으로 ReACT 프롬프트에 다음과 같이 기록됩니다.
```
다음 질문에 최선을 다해 답변하세요. 당신이 사용할 수 있는 도구는 다음과 같습니다.:

ICT_industry_search: This PDF provides information about trends in the ICT industry, including AI.
Calculator: 계산기입니다.

질문: 당신이 답해야 할 입력 질문
생각: 무엇을 해야 할지 생각하세요
행동: 취해야 할 행동을 선택하세요. [ ICT_industry_search, Calculator ] 중 하나여야 합니다.
행동 입력: 행동에 대한 입력을 작성하세요.
관찰: 행동의 결과를 작성하세요
...(이 생각/행동/행동 입력/관찰 단계는 여러 번 반복될 수 있습니다)
생각: 이제 최종 답변을 알겠습니다.
최종 답변: 원래 질문에 대한 최종 답변을 한국어로 작성하세요.

시작하세요!
```

즉 ReACT에서 기본적으로 사용할 Tool에 대한 설명(name과 Description)을 한 후 이 툴을 이용해 **행동**하도록 유도합니다.

In [39]:
prompt_react = hub.pull("hwchase17/react")
print(prompt_react.template)

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


## 모델 선정 및 Agent 생성

In [42]:
llm = ChatOpenAI(model='gpt-4o', temperature=0)

react_agent = create_react_agent(llm, tools=tools, prompt=prompt_react)

react_agent_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [43]:
react_agent_executor.invoke({"input": "세계 인공지능 시장규모를 년도 별로 설명해줘"})



> Entering new AgentExecutor chain...
To answer the question about the global AI market size by year, I will need to search for relevant information in the ICT industry trends document.

Action: ICT_industry_search
Action Input: "global AI market size by year"(40.2)
(48.9)
(32.8)
(49.8)
  
하드웨어
30.9
43.1
60.2
89.2
118.3
176.5
41.7%
　
(39.5)
(39.7)
(48.2)
(32.5)
(49.2)
  
     자료: Global Artificial Intelligence(AI) Market, BCC Research (2022)
￮
인공지능 기술을 도입하는 산업이 늘어나고 있는 점, 인공지능 분야에 진출하는 스타트업의 증가에 
따라 산업의 기술경쟁력이 높아지는 점 등은 시장성장에 촉진요인으로 작용할 전망임
￮
국내 인공지능 기술에 대한 완성도가 높지 않아 국내기술의 도입처가 제한적인 점, 인공지능이

(40.2)
(48.9)
(32.8)
(49.8)
  
하드웨어
30.9
43.1
60.2
89.2
118.3
176.5
41.7%
　
(39.5)
(39.7)
(48.2)
(32.5)
(49.2)
  
     자료: Global Artificial Intelligence(AI) Market, BCC Research (2022)
￮
인공지능 기술을 도입하는 산업이 늘어나고 있는 점, 인공지능 분야에 진출하는 스타트업의 증가에 
따라 산업의 기술경쟁력이 높아지는 점 등은 시장성장에 촉진요인으로 작용할 전망임
￮
국내 인공지능 기술에 대한 완성도가 높지 않아 국내기술의 도입처가 제한적인 점, 인공지능이

세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 
정

{'input': '세계 인공지능 시장규모를 년도 별로 설명해줘',
 'output': 'The global AI market size is projected to grow from $398.4 billion in 2020 to $2,223.7 billion in 2025, with a CAGR of 41.0%.'}

In [44]:
react_agent_executor.invoke({"input": "세계 인공지능 시장규모를 년도 별로 한글로 설명해줘"})



> Entering new AgentExecutor chain...
Action: ICT_industry_search
Action Input: "세계 인공지능 시장규모 년도 별"세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 
정도로 압도적으로 높음
[세계 인공지능 시장규모]
                                                            (단위: 억 달러, 괄호는 YoY %)
구분
2020
2021
2022
2023
2024
2025
CAGR
(2020-2025)
인공지능
398.4
553.3
769.7
1,134.3
1,498.9
2,223.7
41.0%
　
(38.9)
(39.1)
(47.4)
(32.1)
(48.4)
  
소프트웨어
311.8
432.3
600.3
882.4
1,164.6
1,723.5
40.8%
　
(38.6)
(38.8)
(47.0)
(32.0)
(48.0)
  
서비스
55.7
78.0
109.3
162.6
216.0
323.7
42.2%
　
(40.0)

세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 
정도로 압도적으로 높음
[세계 인공지능 시장규모]
                                                            (단위: 억 달러, 괄호는 YoY %)
구분
2020
2021
2022
2023
2024
2025
CAGR
(2020-2025)
인공지능
398.4
553.3
769.7
1,134.3
1,498.9
2,223.7
41.0%
　
(38.9)
(39.1)
(47.4)
(32.1)
(48.4)
  
소프트웨어
311.8
432.3
600.3
882.4
1,164.6
1,723.5
40.8%
　
(38.6)
(38.8)
(47.0)
(32.0)
(48.0)
  
서비스
55.7
78.0
109.3
162.6
216.0
323.7
42.2%
　

{'input': '세계 인공지능 시장규모를 년도 별로 한글로 설명해줘',
 'output': 'Agent stopped due to iteration limit or time limit.'}

In [45]:
from langchain_core.prompts import PromptTemplate

kor_react_prompt_template = """다음 질문에 최선을 다해 답변하세요. 당신이 사용할 수 있는 도구는 다음과 같습니다.:

{tools}

다음 형식을 따릅니다:

Question: 당신이 답해야 할 입력 질문
Thought: 무엇을 해야 할지 생각하세요
Action: 취해야 할 행동을 선택하세요. [{tool_names}] 중 하나여야 합니다.
Action Input: 행동에 대한 입력을 작성하세요.
Observation: 행동의 결과를 작성하세요
...(이 생각/행동/행동 입력/관찰 단계는 여러 번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다.
Final Answer: 원래 질문에 대한 최종 답변을 작성하세요.

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

kor_react_prompt = PromptTemplate.from_template(kor_react_prompt_template)
kor_react_prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template='다음 질문에 최선을 다해 답변하세요. 당신이 사용할 수 있는 도구는 다음과 같습니다.:\n\n{tools}\n\n다음 형식을 따릅니다:\n\nQuestion: 당신이 답해야 할 입력 질문\nThought: 무엇을 해야 할지 생각하세요\nAction: 취해야 할 행동을 선택하세요. [{tool_names}] 중 하나여야 합니다.\nAction Input: 행동에 대한 입력을 작성하세요.\nObservation: 행동의 결과를 작성하세요\n...(이 생각/행동/행동 입력/관찰 단계는 여러 번 반복될 수 있습니다)\nThought: 이제 최종 답변을 알겠습니다.\nFinal Answer: 원래 질문에 대한 최종 답변을 작성하세요.\n\nBegin!\n\nQuestion: {input}\nThought: {agent_scratchpad}')

In [46]:
# Initialize ChatGroq model for language understanding
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Create ReAct agent
react_agent = create_react_agent(llm, tools=tools, prompt=kor_react_prompt) # 한국어 프롬프트 지정

react_agent_executor = AgentExecutor(
    agent=react_agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [47]:
react_agent_executor.invoke({"input": "세계 인공지능 시장규모를 년도 별로 설명해줘"})



> Entering new AgentExecutor chain...
세계 인공지능 시장 규모에 대한 정보를 찾기 위해 ICT 산업 동향 문서를 검색해야 합니다.
Action: ICT_industry_search
Action Input: "세계 인공지능 시장 규모 년도 별"세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 
정도로 압도적으로 높음
[세계 인공지능 시장규모]
                                                            (단위: 억 달러, 괄호는 YoY %)
구분
2020
2021
2022
2023
2024
2025
CAGR
(2020-2025)
인공지능
398.4
553.3
769.7
1,134.3
1,498.9
2,223.7
41.0%
　
(38.9)
(39.1)
(47.4)
(32.1)
(48.4)
  
소프트웨어
311.8
432.3
600.3
882.4
1,164.6
1,723.5
40.8%
　
(38.6)
(38.8)
(47.0)
(32.0)
(48.0)
  
서비스
55.7
78.0
109.3
162.6
216.0
323.7
42.2%
　
(40.0)

세부 솔루션 분문별로는 2020년 기준 소프트웨어 부문의 점유율이 전체시장의 78.3%를 차지할 
정도로 압도적으로 높음
[세계 인공지능 시장규모]
                                                            (단위: 억 달러, 괄호는 YoY %)
구분
2020
2021
2022
2023
2024
2025
CAGR
(2020-2025)
인공지능
398.4
553.3
769.7
1,134.3
1,498.9
2,223.7
41.0%
　
(38.9)
(39.1)
(47.4)
(32.1)
(48.4)
  
소프트웨어
311.8
432.3
600.3
882.4
1,164.6
1,723.5
40.8%
　
(38.6)
(38.8)
(47.0)
(32.0)
(4

{'input': '세계 인공지능 시장규모를 년도 별로 설명해줘',
 'output': '세계 인공지능 시장 규모는 2020년에 398.4억 달러였으며, 2021년에는 553.3억 달러, 2022년에는 769.7억 달러, 2023년에는 1,134.3억 달러, 2024년에는 1,498.9억 달러로 성장할 것으로 예상됩니다. 2025년에는 2,223.7억 달러에 이를 것으로 전망되며, 2020년부터 2025년까지 연평균 성장률(CAGR)은 41.0%로 예상됩니다.'}